# Monitoring report

Builds a drift and performance monitoring report by comparing two windows of scored transactions: feature PSI, score distribution drift, label-rate drift, and bucketed alert volume, precision, recall, and fraud value capture.

**Prerequisites:** scored output, e.g. `make train` followed by `make score-batch`.

In [1]:
from transaction_risk.spark.session import create_spark_session_from_yaml
spark = create_spark_session_from_yaml('../conf/spark.local.yaml')

In [2]:
from pyspark.sql import functions as F

from transaction_risk.spark.io import read_table

scored = read_table(spark, '../data/scored/batch')

# Split the scored table into an earlier reference window and a later current window by time
midpoint = scored.approxQuantile('step', [0.5], 0.001)[0]
reference = scored.filter(F.col('step') <= midpoint)
current = scored.filter(F.col('step') > midpoint)
print('reference rows:', reference.count(), '| current rows:', current.count())

reference rows: 2500 | current rows: 2500


In [3]:
from transaction_risk.monitoring.report import (
    build_monitoring_report,
    write_monitoring_report_json,
    write_monitoring_report_markdown,
)

report = build_monitoring_report(reference, current, feature_columns=['amount'])
write_monitoring_report_json(report, '../reports/monitoring/report.json')
write_monitoring_report_markdown(report, '../reports/monitoring/report.md')

print('labeled mode:', report['labeled'])
print('score PSI:', report.get('score_distribution_psi'))
print('feature PSI:', report.get('feature_drift_psi'))

labeled mode: True
score PSI: 0.010746871675831526
feature PSI: {'amount': 0.0018229357243742723}


The same report is available from the CLI:

```bash
poetry run transaction-risk monitor \
  --reference data/scored/reference \
  --current data/scored/current \
  --output-json reports/monitoring/report.json \
  --output-md reports/monitoring/report.md
```

In [4]:
spark.stop()